# 07 · The four modes, one at a time (offline)

Each cell tests one mode's models on a still picture, so you can see what each
model says before using it live. Run `tools/download_models.py` once with
internet; after that this works with Wi-Fi off.

In [ ]:
import sys, pathlib
LAB = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(LAB))
from va_lab import config
config.ensure_dirs()
print("lab folder:", LAB)

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"
import cv2, matplotlib.pyplot as plt
from va_lab import understand, ocr, safety
from va_lab.vision import open_camera, grab
cap = open_camera(); frame = grab(cap); cap.release()
plt.figure(figsize=(7, 5)); plt.imshow(frame[..., ::-1]); plt.axis("off"); plt.show()

## Describe: Florence-2 caption → NLLB Bangla → safety filter

In [ ]:
en = understand.caption(frame, "detailed")
bn = understand.to_bangla(en)
print("EN:", en); print("BN:", bn); print("after safety filter:", safety.clean(bn))

## Identify: the middle of the picture

In [ ]:
h, w = frame.shape[:2]
crop = frame[int(h*.2):int(h*.8), int(w*.25):int(w*.75)]
print("name:", understand.caption(crop, "short"), "| colour:", understand.colour_name(crop))
plt.imshow(crop[..., ::-1]); plt.axis("off"); plt.show()

## Read: hold a printed page in front of the camera, then run

In [ ]:
cap = open_camera(); page = grab(cap, warmup=10); cap.release()
reader = understand.get("ocr")
print("aiming hint:", ocr.check(page, reader.boxes(page)))
res = reader.read(page)
for para in ocr.reading_order(res):
    print("¶", para)
    for s in ocr.sentences(para): print("   -", s)

## Find any object by name (YOLO-World, open vocabulary)

In [ ]:
from va_lab.vision import Detector
det = Detector("world")
query, said = understand.query_for("চাবি")      # Bangla name -> English query
print(said, "->", query)
det.set_queries([query], "target")
print([(d.obj, round(d.angle_deg), round(d.distance_m, 1)) for d in det.detect(frame)])
det.restore()